# Analiz: tablolar ve makale grafikleri

Bu notebook **yalnızca `outputs/` klasörünü** okur; ham veriye ve modellere ihtiyaç duymaz.
Arkadaşların sonuçlarını GitHub'a yükledikçe tablolar ve grafikler üç veri setini de kapsar.

- **VS Code:** Sağ üstten çekirdek olarak `iot-security` ortamını seç → *Run All*.
- **Colab:** GitHub'dan aç (*File → Open notebook → GitHub*) → ilk hücre repoyu çeker.

Grafikler `figures/` klasörüne PNG olarak (200 dpi) kaydedilir.

In [ ]:
# Yalnızca Colab'da: repoyu çek (Secrets'ta GITHUB_TOKEN gerekli). VS Code'da bir şey yapmaz.
import os, sys, subprocess
if 'google.colab' in sys.modules:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if not os.path.exists('/content/bitirme-projesi'):
        subprocess.run(['git', 'clone', '-q', f'https://{token}@github.com/GamzeSerin/bitirme-projesi.git',
                        '/content/bitirme-projesi'], check=True)
    os.chdir('/content/bitirme-projesi/notebooks')
    subprocess.run(['git', 'pull', '-q'], check=True)

In [ ]:
import glob, os, sys
ROOT = os.path.abspath('..') if os.path.exists(os.path.join('..', 'main.py')) else os.getcwd()
sys.path[:0] = [ROOT, os.path.join(ROOT, 'src')]

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

import main as dss
from common import contract, report

report.setup_style()
pd.set_option('display.precision', 4)

OUTPUTS = os.path.join(ROOT, 'outputs')
has_results = lambda d: any(os.path.exists(os.path.join(x, 'manifest.json')) for x in glob.glob(os.path.join(d, '*')))
if not has_results(OUTPUTS) and has_results(os.path.join(ROOT, 'outputs_quick')):
    OUTPUTS = os.path.join(ROOT, 'outputs_quick')
    print('⚠ outputs/ henüz boş: HIZLI DENEME sonuçları gösteriliyor (makale için değil).')
FIG_DIR = os.path.join(ROOT, 'figures')

envs = {}
for d in sorted(glob.glob(os.path.join(OUTPUTS, '*'))):
    if not os.path.isdir(d):
        continue
    errors = contract.validate(d)
    if errors:
        print(f'✗ {os.path.basename(d)} atlandı: {errors}')
        continue
    env = dss.EnvironmentModel(d)
    envs[env.id] = env

pd.DataFrame({eid: {'veri seti': e.manifest.get('dataset_name'), 'eğitim': e.manifest['n_train'],
                    'test': e.manifest['n_test'], 'birincil model': e.manifest['primary_model'],
                    'DSS oylayanlar': ', '.join(e.voters)} for eid, e in envs.items()}).T

## 1. Model karşılaştırması
Ana metrik **macro-F1**. Birincil model OOF'a göre seçilir; test skoru yalnızca raporlama içindir.

In [ ]:
summary = report.model_summary(envs)
os.makedirs(FIG_DIR, exist_ok=True)
summary.to_csv(os.path.join(FIG_DIR, 'table_model_summary.csv'))
summary

In [ ]:
dss_eval = {eid: dss.evaluate_environment(e) for eid, e in envs.items()}
sample_scores = {eid: r['macro_f1_on_test_sample'] for eid, r in dss_eval.items()}
fig, path = report.plot_model_comparison(envs, FIG_DIR, sample_scores)
plt.show()
print('Modeller ve DSS test_predictions dosyasındaki aynı satırlarda ölçülür (varsayılan: tüm test seti).')

## 2. Saldırı tipi × model (makalenin ana sorusu)

In [ ]:
for eid, env in envs.items():
    display(Markdown(f'### {eid}'))
    display(report.best_model_table(env))
    fig, _ = report.plot_per_class_heatmap(env, FIG_DIR)
    plt.show()

## 3. Hangi saldırılar karışıyor?

In [ ]:
for eid, env in envs.items():
    fig, _ = report.plot_confusion(env, FIG_DIR)
    plt.show()
    fig, _ = report.plot_fine_label_recall(env, FIG_DIR)
    if fig:
        plt.show()

## 3b. Güven aralıkları (bootstrap)
Test satırları 500 kez yeniden örneklenir. İki modelin %95 aralıkları çakışıyorsa aralarındaki fark anlamlı sayılmaz.
Az örnekli sınıflarda (WEB, BRUTEFORCE) aralıklar geniştir; makalede F1 bu aralıklarla birlikte verilmelidir.

In [ ]:
for eid, env in envs.items():
    display(Markdown(f'### {eid}'))
    macro_ci, class_ci = report.bootstrap_ci(env, dss.dss_predict)
    display(macro_ci)
    display(class_ci)
    macro_ci.to_csv(os.path.join(FIG_DIR, f'table_macro_f1_ci_{eid}.csv'))
    class_ci.to_csv(os.path.join(FIG_DIR, f'table_class_f1_ci_{eid}.csv'))

## 4. NIDS bakışı: saldırı var mı?
BENIGN dışındaki her tahmin bir **alarm** sayılır. Tespit oranı (DR) ve yanlış alarm oranı (FPR),
gerçek ağlardaki saldırı oranından bağımsızdır. Gerçek ağlarda trafiğin büyük çoğunluğu normal olduğu için FPR kritiktir.

In [ ]:
for eid, env in envs.items():
    display(Markdown(f'### {eid}'))
    display(report.nids_binary(env, dss.dss_predict))

## 5. Karar Destek Sistemi (DSS)

In [ ]:
for eid, r in dss_eval.items():
    display(Markdown(f"### {eid}: oylayan modeller {', '.join(r['dss_voters'])} · "
                     f"ALLOW edilen saldırı: **{r['attacks_allowed']}** / {r['n_rows']} satır"))
    fig, _ = report.plot_dss_actions(envs[eid], dss.dss_predict, FIG_DIR)
    plt.show()

## 6. Açıklanabilirlik (SHAP)
Sınıf başına en önemli özellikler. Beeswarm grafikleri eğitim sırasında `outputs/<veri seti>/figures/` altına kaydedilir.

In [ ]:
for eid, env in envs.items():
    if not env.shap:
        continue
    display(Markdown(f"### {eid} · {env.shap['model']}"))
    display(pd.DataFrame({c: [f['feature'] for f in v[:5]] for c, v in env.shap['per_class'].items()},
                         index=[f'#{i}' for i in range(1, 6)]))
    bar = os.path.join(env.dir, 'figures', 'shap_global_bar.png')
    if os.path.exists(bar):
        display(Image(bar, width=600))

## 7. Veri setleri arası karşılaştırma
En az iki veri setinin sonuçları yüklendiğinde dolar.

In [ ]:
fig, _ = report.plot_cross_dataset(envs, FIG_DIR)
if fig:
    plt.show()
else:
    print('Şimdilik tek veri seti var: arkadaşlarının sonuçları geldikçe bu bölüm dolacak.')

## 8. Deney ortamı (makale için)

In [ ]:
pd.DataFrame({eid: {**{k: v for k, v in e.manifest.get('runtime', {}).items() if k != 'libraries'},
                    **e.manifest.get('runtime', {}).get('libraries', {})} for eid, e in envs.items()})